In [2]:
# EX 00

def sum2(a, b):
    if not b:
        return a
    return sum2(a ^ b, (a & b) << 1)

print('--------------- Sumtests ------------------')
print(sum2(140,2025), sum([140, 2025]))
print(sum2(99999, 2), sum([99999, 2]))

--------------- Sumtests ------------------
2165 2165
100001 100001


In [3]:
# EX 01

def mul2(a, b):
    res = 0
    while b:
        if b & 1:
            res = sum2(res, a)
        a = a << 1
        b = b >> 1
    return res

print('--------------- Multests ------------------')
print(mul2(10,23), 10*23)
print(mul2(400,25), 400*25)

--------------- Multests ------------------
230 230
10000 10000


In [26]:
# EX 02

def grayCode(a):
    return a ^ (a >> 1)

print('--------------- Graytests -----------------')
print('format: (value, converted value)')
print(*[(f, grayCode(f)) for f in range(0,32)], sep=', \n')

--------------- Graytests -----------------
format: (value, converted value)
(0, 0), 
(1, 1), 
(2, 3), 
(3, 2), 
(4, 6), 
(5, 7), 
(6, 5), 
(7, 4), 
(8, 12), 
(9, 13), 
(10, 15), 
(11, 14), 
(12, 10), 
(13, 11), 
(14, 9), 
(15, 8), 
(16, 24), 
(17, 25), 
(18, 27), 
(19, 26), 
(20, 30), 
(21, 31), 
(22, 29), 
(23, 28), 
(24, 20), 
(25, 21), 
(26, 23), 
(27, 22), 
(28, 18), 
(29, 19), 
(30, 17), 
(31, 16)


In [15]:
# Python's builtin bitshift preserves sign bit :( (eg -1 >> 1 = -1)
# Real logical difference should have the following implementation:
#   def sub(a,b):
#       return sum2(a, sum2(~b, 1))
# which is the regular sum of a and b's two's complement
def sub2(a, b):
    return a - b


# EX 03
dic = {'1': (lambda a, b: True),
       '0': (lambda a, b: False),
       '!': (lambda a, b: not a.evl()),
       '&': (lambda a, b: a.evl() & b.evl()),
       '|': (lambda a, b: a.evl() | b.evl()),
       '^': (lambda a, b: a.evl() ^ b.evl()),
       '>': (lambda a, b: (not a.evl()) | b.evl()),
       '=': (lambda a, b: a.evl() == b.evl()),
       None: lambda a, b: ...}

silent = True


class LogiNode:
    def __init__(self, tipe, a=None, b=None):
        self._a = a
        self._b = b
        self._tipe = dic[tipe]
        self._t = tipe

    def evl(self):
        global silent
        if not silent:
            print('evaluating an', self._t)
        return self._tipe(self._a, self._b)


i = 0

# this is NOT the traditional reverse polish parser:
# V2 will fix this issue by using a stack instead of recursion;
# still this is an equivalent implementation (resursion is actually stack hehe).
def parseInner(s):
    global i
    i = sub2(i, 1)
    if s[i] == '1' or s[i] == '0':
        return LogiNode(s[i])
    if s[i] == '!':
        return LogiNode(s[i], parseInner(s))
    return LogiNode(s[i], parseInner(s), parseInner(s))


def parse(s):
    global i
    i = len(s)
    return parseInner(s)

print('--------------- Parsetests ----------------')
print('1!01^&!!!! is', parse("1!01^&!!!!").evl())
print('1!01||!! is', parse("1!01||!!").evl())
print('0!0|1&! is', parse("0!0|1&!").evl())
print('0!0|1&!! is', parse("0!0|1&!!").evl())

--------------- Parsetests ----------------
1!01^&!!!! is False
1!01||!! is True
0!0|1&! is False
0!0|1&!! is True


In [ ]:
# EX 04, 07
# Bit to int for printing purposes
def btoi(b):
    return 1 if b else 0

# Is a '1' or a '0'
def isbit(c):
    return c == '0' or c == '1' or c.isupper()

# Is a binary operator
def isbin(c):
    return c in ['|', '&', '^', '=', '>']


class ParserV2:
    silent = True
    title = ''
    s = ''
    end = None
    variables = {}
    sat_test = None

    @staticmethod
    def parseTipe(t):
        match t:
            case '1': return (lambda a, b: True)
            case '0': return (lambda a, b: False)
            case '!': return (lambda a, b: not a.evl())
            case '&': return (lambda a, b: a.evl() & b.evl())
            case '|': return (lambda a, b: a.evl() | b.evl())
            case '^': return (lambda a, b: a.evl() ^ b.evl())
            case '>': return (lambda a, b: (not a.evl()) | b.evl())
            case '=': return (lambda a, b: a.evl() == b.evl())
            case t if t.isupper():
                p = ParserV2.Variable.getVar(t)
                return (lambda a, b: p.evl())
            case None: lambda a, b: ...
            case _: print('bad input!')

    class Variable:
        def __init__(self, name):
            ParserV2.variables[name] = self
            self.name = name
            self.val = False

        def setVal(self, val):
            self.val = val

        def notVal(self):
            self.val = not self.val

        def evl(self):
            return self.val

        @classmethod
        def getVar(v, name):
            if name in ParserV2.variables:
                return ParserV2.variables[name]
            else:
                return v(name)

    class LogiNodeV2:
        def __init__(self, tipe, a=None, b=None):
            self._a = a
            self._b = b
            self._tipe = ParserV2.parseTipe(tipe)
            self._t = tipe

        def evl(self):
            if not ParserV2.silent:
                print('evaluating an', self._t, (self._tipe(
                    None, None) if self._t.isupper() else ''))
            return self._tipe(self._a, self._b)
    LogiNode = LogiNodeV2

    @classmethod
    def parseInner(pv2):
        i = 0
        stack = []
        while sub2(i, len(pv2.s)):
            match pv2.s[i]:
                case ' ':
                    ...
                case t if isbit(t):
                    stack.append(pv2.LogiNode(t))
                case '!':
                    stack.append(pv2.LogiNode(t, stack.pop()))
                case _:
                    stack.append(pv2.LogiNode(t, stack.pop(), stack.pop()))
            i = sum2(i, 1)
        if sub2(len(stack), 1):
            print('parse error!', stack)
        return stack[0]

    @classmethod
    def parse(pv2, s, title=''):
        pv2.title = title
        pv2.s = s
        pv2.variables.clear()
        pv2.end = pv2.parseInner()
        return pv2

    @classmethod
    def evl(pv2):
        return pv2.end.evl()

    @classmethod
    def gets(pv2):
        return pv2.s

    @classmethod  # Gray style variables permutation running
    def printSubTable(pv2, currVarIndex, silent=False):
        if currVarIndex == len(pv2.variables):
            e = pv2.evl()
            if not silent:
                print('', *[btoi(v.evl()) for v in pv2.variables.values()],
                      btoi(e), '', sep=' | ')
            if e:
                pv2.sat_test = True
            return
        currVarName = list(pv2.variables)[currVarIndex]
        pv2.variables[currVarName].setVal(False)
        pv2.printSubTable(sum2(currVarIndex, 1), silent)
        pv2.variables[currVarName].setVal(True)
        pv2.printSubTable(sum2(currVarIndex, 1), silent)

    @classmethod
    def printTable(pv2, silent=False):
        pv2.variables = dict(sorted(pv2.variables.items()))
        if not silent:
            print('  Printing truth table for "', pv2.gets(), '"',
                  (' (' + pv2.title + ')' if pv2.title else ''), sep='')
        if not silent:
            print('', *pv2.variables.keys(), '= |', sep=' | ')
        pv2.printSubTable(0, silent)

    @classmethod
    def satTest(pv2):
        pv2.sat_test = False
        pv2.printTable(True)
        return pv2.sat_test

print('--------------- ParseV2tests ----------------')
ParserV2.parse('BAA!>=').printTable()
ParserV2.parse('A!B!|AB&!=', 'De Morgan Law').printTable()
ParserV2.parse('CB|CA|&CBA&|=', 'Distributivity').printTable()
ParserV2.parse('CB& CA& |  C BA| &   =', 'Other Distributivity with whitespace').printTable()

--------------- ParseV2tests ----------------
  Printing truth table for "BAA!>="
 | A | B | = |
 | 0 | 0 | 1 | 
 | 0 | 1 | 0 | 
 | 1 | 0 | 0 | 
 | 1 | 1 | 1 | 
  Printing truth table for "A!B!|AB&!=" (De Morgan Law)
 | A | B | = |
 | 0 | 0 | 1 | 
 | 0 | 1 | 1 | 
 | 1 | 0 | 1 | 
 | 1 | 1 | 1 | 
  Printing truth table for "CB|CA|&CBA&|=" (Distributivity)
 | A | B | C | = |
 | 0 | 0 | 0 | 1 | 
 | 0 | 0 | 1 | 1 | 
 | 0 | 1 | 0 | 1 | 
 | 0 | 1 | 1 | 1 | 
 | 1 | 0 | 0 | 1 | 
 | 1 | 0 | 1 | 1 | 
 | 1 | 1 | 0 | 1 | 
 | 1 | 1 | 1 | 1 | 
  Printing truth table for "CB& CA& |  C BA| &   =" (Other Distributivity with whitespace)
 | A | B | C | = |
 | 0 | 0 | 0 | 1 | 
 | 0 | 0 | 1 | 1 | 
 | 0 | 1 | 0 | 1 | 
 | 0 | 1 | 1 | 1 | 
 | 1 | 0 | 0 | 1 | 
 | 1 | 0 | 1 | 1 | 
 | 1 | 1 | 0 | 1 | 
 | 1 | 1 | 1 | 1 | 


In [21]:
# EX 05, 06, 07
class ParserV3(ParserV2):
    silent = True
    use_NNF = True
    use_CNF = True
    use_DNF = use_CNF or True
    simplify_ops = use_DNF or False

    class LogiNodeV3(ParserV2.LogiNodeV2):
        def getArgs(self):
            return [self._a, self._b, self._t]

        def reset(self, tipe, a=None, b=None):
            self.__init__(tipe, a, b)
            return self

        def swap(self):
            return self.reset(self._t, self._b, self._a)

        def negate(self):
            match self._t:
                case '!': return self._a
                case '&': return self.reset('|', self._a.negate(), self._b.negate())
                case '|': return self.reset('&', self._a.negate(), self._b.negate())
                case '>': return self.reset('&', self._a, self._b.negate())
                case '=': return self.reset('=', self._a, self._b.negate())
                case '^': return self.reset('^', self._a, self._b.negate())
                case _: return ParserV3.LogiNode('!', self)

        def __str__(self):
            match self._t:
                case '!': return '!' + str(self._a)
                case '&' | '^' | '|' | '>' | '=': return self._t + str(self._a) + str(self._b)
                case _: return self._t

        def dup(self):
            return ParserV3.LogiNode(self._t, self._a.dup() if self._a else None, self._b.dup() if self._b else None)

        def simplify(self):
            ParserV3.steps = sum2(ParserV3.steps, 1)
            # Propagate negation (de Morgan etc...)
            if ParserV3.use_NNF and self._t == '!' and not isbit(self._a._t):
                return self._a.negate().simplify()
            # Expand non-connective operators
            if ParserV3.simplify_ops:
                if self._t == '=':
                    self.reset('|', ParserV3.LogiNode('&', self._a, self._b), ParserV3.LogiNode(
                        '&', self._a.dup().negate(), self.dup()._b.negate()))
                    ParserV3.redo = True
                if self._t == '^':
                    self.reset('=', self._a, self._b).negate().simplify()
                    ParserV3.redo = True
                if self._t == '>':
                    self.reset('|', self._a, self._b.negate())
                    ParserV3.redo = True
            # Special simplifications
            if ParserV3.use_DNF:
                if isbin(self._t):
                    if not ParserV3.silent:
                        print(self._t, self._a._t, self._b._t)
                # (A | B) | (C | D) -> A | (B | (C | D)) (commute)
                if self._t == '|' == self._a._t == self._b._t:
                    # print('MATCH | | |')
                    temp = self._a._a
                    self._a._a = self._b
                    self._b = temp
                    ParserV3.redo = True
                # AB|C| -> ABC|| (operand swap)
                if isbin(self._t) and self._b._t == '|':
                    # print('swapping since left is |')
                    self.swap()
                # (A | B) & C -> (A & C) | (B & C). This causes the parsing process to restart since the LogiNode tree is one-way
                if self._t == '&':
                    if self._a._t == '|':
                        self.reset('|', ParserV3.LogiNode('&', self._a._a, self._b), ParserV3.LogiNode(
                            '&', self._a._b, self._b.dup()))
                        ParserV3.redo = True
            # Recurse
            if self._a:
                self._a = self._a.simplify()
            if self._b:
                self._b = self._b.simplify()
            return self
    LogiNode = LogiNodeV3

    @classmethod
    def simplify(pv3):
        pv3.steps = 0
        pv3.redo = True
        i = 0
        while pv3.redo:
            if not pv3.silent:
                print('-- step --', i)
            i = sum2(i, 1)
            pv3.redo = False
            if pv3.use_CNF:
                pv3.end = pv3.end.negate().simplify().negate()
            else:
                pv3.end = pv3.end.simplify()
        pv3.s = pv3.deparse()
        return pv3.steps

    @classmethod
    def deparse(pv3):
        return str(pv3.end)[::-1]

print('--------------- ParseV3 tests ----------------')
ParserV3.parse('A!B!|AB&!=', 'De Morgan I').printTable()
print('Simplification rounds count:', ParserV3.simplify())
ParserV3.printTable()
print(ParserV3.satTest())

--------------- ParseV3 tests ----------------
  Printing truth table for "A!B!|AB&!=" (De Morgan I)
 | A | B | = |
 | 0 | 0 | 1 | 
 | 0 | 1 | 1 | 
 | 1 | 0 | 1 | 
 | 1 | 1 | 1 | 
Simplification rounds count: 105
  Printing truth table for "A!B!|B|A!B!|A|A!B!|A|A!B!|B|&&&" (De Morgan I)
 | A | B | = |
 | 0 | 0 | 1 | 
 | 0 | 1 | 1 | 
 | 1 | 0 | 1 | 
 | 1 | 1 | 1 | 
True


In [31]:
# EX 08
def msb(n):
    i = 0
    if n < 0:
        n = sub2(0, 2)
    while n:
        n = n >> 1
        i = sum2(i, 1)
    return i

# evil python iterator + bitmask manipulation
def powerset(v):
    for mask in range(0, pow(2, len(v))):
        yield [v[i] for i in range(0, msb(mask)) if mask & 1 << i]

print('--------------- Powerset tests ----------------')
print(*list(powerset([1, 11, 23, 19, 100])), sep=', \n')

--------------- Powerset tests ----------------
[], 
[1], 
[11], 
[1, 11], 
[23], 
[1, 23], 
[11, 23], 
[1, 11, 23], 
[19], 
[1, 19], 
[11, 19], 
[1, 11, 19], 
[23, 19], 
[1, 23, 19], 
[11, 23, 19], 
[1, 11, 23, 19], 
[100], 
[1, 100], 
[11, 100], 
[1, 11, 100], 
[23, 100], 
[1, 23, 100], 
[11, 23, 100], 
[1, 11, 23, 100], 
[19, 100], 
[1, 19, 100], 
[11, 19, 100], 
[1, 11, 19, 100], 
[23, 19, 100], 
[1, 23, 19, 100], 
[11, 23, 19, 100], 
[1, 11, 23, 19, 100]


In [ ]:
# EX 09
class SetParser(ParserV3):
    env = []

    @staticmethod
    def parseTipe(t):
        match t:
            case '!': return (lambda a, b: [v for v in SetParser.env if v not in a.evl()])
            case '~': return (lambda a, b: [v for v in a.evl() if v not in b.evl()])
            case '&': return (lambda a, b: [v for v in a.evl() if v in b.evl()])
            case '|': return (lambda a, b: [v for s in [a.evl(), b.evl()] for v in s])
            case '^': return (lambda a, b: (sa := a.evl(), sb := b.evl(), [v for s in [sa, sb] for v in s if not (v in sa and v in sb)])[2])
            case t if t.isupper():
                p = SetParser.SetVariable.getVar(t)
                return (lambda a, b: p.evl())
            case None: lambda a, b: ...
            case _: print('bad input!')

    class SetNode:
        def __init__(self, tipe, a=None, b=None):
            self._a = a
            self._b = b
            self._tipe = SetParser.parseTipe(tipe)
            self._t = tipe

        def evl(self):
            if not ParserV2.silent:
                print('evaluating an', self._t, (self._tipe(
                    None, None) if self._t.isupper() else ''))
            return self._tipe(self._a, self._b)
    LogiNode = SetNode

    class SetVariable:
        def __init__(self, name):
            SetParser.variables[name] = self
            self.name = name
            self.val = False

        def setVal(self, val):
            self.val = val

        def notVal(self):
            self.val = not self.val

        def evl(self):
            return self.val

        @classmethod
        def getVar(v, name):
            if name in SetParser.variables:
                return SetParser.variables[name]
            else:
                return v(name)

    def printTable():  # deactivated
        ...

    def satTest():     # deactivated
        ...

    @classmethod
    def parse(sp, s):
        sp.s = s
        sp.variables.clear()
        sp.end = sp.parseInner()

    @classmethod
    def evl(sp, sets, env=[]):
        # print(sp.variables)
        sp.variables = dict(sorted(sp.variables.items()))
        sp.env = [v for s in sets for v in s] + env
        for i, (n, v) in enumerate(sp.variables.items()):
            print('Parsing', sets[i], 'as', n)
            v.setVal(sets[i])
        return list(set(sp.end.evl()))

print(' ')
print(' ')
print(' ')
print('-------------- SetParsertests ---------------')

SetParser.parse('AA|BC|&')
print(SetParser.evl([[2,3,4,5], [1,4], [999]]))

 
 
 
-------------- SetParsertests ---------------
Parsing [2, 3, 4, 5] as A
Parsing [1, 4] as B
Parsing [999] as C
[4]


In [94]:
# EX 10, 11
# Cantor Diagonal method (custom)
def _sfc(x, y): # N^2 -> N
    r = ((x + y + 1) ** 2 + x - y - 1) // 2
    # print(r)
    return r


def _antiSfc(r): # N -> N^2
    # print("processing an", r)
    c = (pow(1 + 8 * r, 0.5) - 1) // 2 + 1
    ct = (c ** 2 + c) // 2
    return (int(c + r - ct), int(ct - r - 1))

In [101]:
# Bit interwinding method
def _sfc(a,b): # N^2 -> N
    res = 0
    i = 0
    while a or b:
        res |= (a & 1) << i
        a >>= 1
        i = sum2(i, 1)
        res |= (b & 1) << i
        b >>= 1
        i = sum2(i, 1)
        # print(res, a ,b)
    return res

def _antiSfc(c): # N -> N^2
    a = 0
    b = 0
    i = 0
    while c:
        a |= (c & 1) << i
        c >>= 1
        b |= (c & 1) << i
        c >>= 1
        i = sum2(i, 1)
        # print(c, a ,b)
    return (a,b)

In [ ]:
def sfc(a, b):
    t = _sfc(a, b)
    # proceed with mapping [0, INT_MAX] ∩ N -> [0, 1]
    # triangular simplification: if (a,b) is on the top right of the [0, INT_MAX]^2 grid
    # then 'fold' the point and map to [0.5, 1], otherwise map to [0, 0.5[
    if t > (1 << 31):
        return 1 - _sfc(sub2((1 << 16), sum2(a, 1)), sub2((1 << 16), sum2(b, 1))) / (1 << 32)
    else:
        return t / (1 << 32)

def antiSfc(r):
    # proceed with mapping [0, 1] -> [0, INT_MAX] ∩ N
    # unpack + unfold
    if r > 0.5:
        (a, b) = _antiSfc(int((1 - r) * (1 << 32)))
        return (sub2((1 << 16), sum2(a, 1)), sub2((1 << 16), sum2(b, 1)))
    else:
        return _antiSfc(int(r * (1 << 32)))


# testing
def doLongTest(v, silent = True):
    if not silent:
        print("long testing:", v[0:20], f"... ({len(v)}) in total" if len(v) > 20 else "")
    test = [i for i in v if i != antiSfc(sfc(*i))]
    if len(test) != 0:
        print('test failed for', test)
    else:
        print('test succesful!')

def doLongInverseTest(v, silent = True):
    if not silent:
        print("long testing:", v[0:20], f"... ({len(v)}) in total" if len(v) > 20 else "...")
    test = [i for i in v if i != sfc(*antiSfc(i))]
    if len(test) != 0:
        print('test failed for', test)
    else:
        print('test succesful!')

def doVerboseTest(a,b):
    r = sfc(a,b)
    (x,y) = antiSfc(r)
    print("Converted (", a, ', ', b,') into ', r, ' and then into (', x, ', ', y, '): ', ('Match!' if (x,y) == (a,b) else 'No match :('), sep='')

def doVerboseInverseTest(r):
    (x,y) = antiSfc(r)
    s = sfc(x,y)
    print("Converted ", r, " into (", x, ', ', y, ') and then into ', s,': ', ('Match!' if r == s else 'No match :( they convert into ', sfc(x,y)), sep='')

print('----------------- SFC tests ------------------')
# this will test all numbers close (+/- 10) to all powers of two from 0 to 16 (only positive), couples are generated by any combination of said numbers. "set()" reduces redundance
t = set([b for b in ([(1 << i) + c for c in list(range(-10, 10)) for i in range(0, 16) if (1 << i) + c >= 0])])
doLongTest([(a, b) for a in t for b in t] + [(55606, 37075), (0,0), ((1<<16) - 1, (1<<16) - 1)], False)
doLongInverseTest([(1 << i) / (1 << 16) for i in range(0, 16)], False)

# doVerboseTest(32767, 32768)abs
([doVerboseTest(32768 + i, 32767 - i) for i in range(-20, 19)])
doVerboseInverseTest(0.5)
print(antiSfc(0.2222))

----------------- SFC tests ------------------
long testing: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (0, 11), (0, 12), (0, 1020), (0, 13), (0, 509), (0, 14), (0, 1021), (0, 15), (0, 511)] ... (58567) in total
test succesful!
long testing: [1.52587890625e-05, 3.0517578125e-05, 6.103515625e-05, 0.0001220703125, 0.000244140625, 0.00048828125, 0.0009765625, 0.001953125, 0.00390625, 0.0078125, 0.015625, 0.03125, 0.0625, 0.125, 0.25, 0.5] ...
test succesful!
Converted (32748, 32787) into 0.5833333942573518 and then into (32748, 32787): Match!
Converted (32749, 32786) into 0.5833333940245211 and then into (32749, 32786): Match!
Converted (32750, 32785) into 0.5833333933260292 and then into (32750, 32785): Match!
Converted (32751, 32784) into 0.5833333930931985 and then into (32751, 32784): Match!
Converted (32752, 32783) into 0.5833333532791585 and then into (32752, 32783): Match!
Converted (32753, 32782) into 0.5833333530463278 and then into 